## 移动通信的功率分配问题 -- 本文件为配置文件
配置文件中定义了场景的物理参数，包括载波频率、光速、接收机偏移距离等。

### 场景几何：直线路 + 静止偏移接收机 （SISO -- single input single output）

下图说明 pass-by episode 的几何结构。
**关键点：道路是笔直直线（车匀速从左到右驶过），'U 形'指车辆-接收机距离 $d(t)$ 随时间的变化曲线，而非道路形状。**

![pass-by scene geometry](figures/scene_geometry.png)

- 左（俯视图）：接收机静止在道路旁，横向偏移 $d_\perp = 20$ m；蓝色虚线是以接收机为圆心的等距环，橙色点线是上行波前（以发射车辆为圆心）。
- 右（距离曲线）：$d(t)=\sqrt{d_\perp^2 + x(t)^2}$，$x(t)=v(t-T/2)$；三档速度对应深浅不同的对称 U 形，最近点均在 $t=T/2$。

绘图脚本：`figures/plot_scene.py`（修改 `SceneConfig` 参数后重跑即可重新生成 PDF/PNG）。

In [1]:
""" 
信道生成(阶段A解析/阶段B Sionna)、Gym环境、基线、评估共用本配置。
"""

import numpy as np
from dataclasses import dataclass

## 载波频率 $f_c = 2.0\ \mathrm{GHz}$ 的选择依据

**1. 对标真实高铁蜂窝部署。** 高铁沿线移动覆盖主要工作在 Sub-6 GHz 中低频段
（4G LTE Band 1/3 的 $1.8/2.1\ \mathrm{GHz}$，5G 高铁专网 n1 频段 $2.1\ \mathrm{GHz}$）。
$2.0\ \mathrm{GHz}$ 位于该区间中心，是高铁文献最常用的仿真频点之一。

**2. 物理尺度与本实验的时隙结构匹配。** 载波频率在模型中决定三个量：

$$\lambda = \frac{c}{f_c}, \qquad
L_{\mathrm{fs}}(d) = 20\log_{10}\!\left(\frac{4\pi d\, f_c}{c}\right)\ \mathrm{dB}, \qquad
f_D = \frac{v\, f_c}{c}$$

> **概念补充 — 自由空间路径损耗 $L_{\mathrm{fs}}(d)$**
>
> 由 **Friis 传输方程**导出：发射功率 $P_t$ 经理想全向天线后，功率以球面波形式扩散，
> 接收功率密度按 $1/d^2$ 衰减：
> $$P_r = P_t\, G_t G_r\!\left(\frac{\lambda}{4\pi d}\right)^2
> \;\Longrightarrow\;
> L_{\mathrm{fs}}(d)=\frac{P_t}{P_r}\bigg|_{G_t=G_r=1}
> =\left(\frac{4\pi d}{\lambda}\right)^2
> =\left(\frac{4\pi d f_c}{c}\right)^2$$
> 写成 dB 即 $L_{\mathrm{fs}}^{\mathrm{dB}}=20\log_{10}(4\pi d f_c/c)$。
> 含义：**距离每增大 10 倍损耗 +20 dB；频率每升高 10 倍也 +20 dB**
> （高频损耗大并非"空气吸收"，而是固定口径天线对更短波长的有效面积 $\propto\lambda^2$）。
> 本实验把大尺度信道增益取为 $g_{\mathrm{large}}=10^{-L_{\mathrm{fs}}(d)/10}$，
> 再乘阴影衰落与 TDL 小尺度衰落，得到轨迹文件中的线性增益 $g_t$。

> **概念补充 — 多普勒频移 $f_D$ 与信道的时间选择性**
>
> 收发两端存在径向相对速度 $v$ 时，波前被"压缩/拉伸"，接收频率相对载波偏移
> $$f_D = \frac{v}{\lambda} = \frac{v f_c}{c},$$
> 驶近时为正、驶离时为负。本实验车辆直线驶过横向偏移 $d_\perp=20$ m 的静止节点，
> 径向速度 $v_{\mathrm{rad}}(t)=v\,x(t)/d(t)$ 沿 **U 形轨迹由正连续变负、过最近点时为零**，
> 故瞬时多普勒在一次经过中过零一次。
> 多普勒决定信道的**时间选择性（变化快慢）**：多径各方向的多普勒展宽构成谱宽 $\sim f_D$，
> 对应的**相干时间**（信道近似不变的时间跨度）约为
> $$T_c \sim \frac{1}{2f_D}\quad(\text{数量级估计，文献常用 }0.423/f_D),$$
> 而瑞利衰落复信道的时间自相关为 $R_h(\tau)=J_0(2\pi f_D\tau)$，
> **功率增益**自相关为其平方 $R_g(\tau)=J_0^2(2\pi f_D\tau)$——
> 这正是下表用 $J_0^2$ 预测 lag-1 相关性、并用 $f_D\Delta t$ 衡量
> "相邻时隙信道还像不像"的理论依据。

其中最关键的是**最大多普勒频移 $f_D$**，它与决策时隙 $\Delta t = 1\ \mathrm{ms}$ 的无量纲乘积
$f_D \Delta t$ 决定了时隙间信道的相关程度：

| 速度 $v$ | $\lambda = c/f_c$ | $f_D = v f_c/c$ | $f_D \Delta t$ |
|---|---|---|---|
| $30\ \mathrm{km/h}$ | $0.15\ \mathrm{m}$ | $55.6\ \mathrm{Hz}$ | $0.056$ |
| $120\ \mathrm{km/h}$ | $0.15\ \mathrm{m}$ | $222.2\ \mathrm{Hz}$ | $0.222$ |
| $350\ \mathrm{km/h}$ | $0.15\ \mathrm{m}$ | $648.1\ \mathrm{Hz}$ | $0.648$ |

三档速度恰好横跨"强相关 → 去相关"区间（实测 lag-1 功率增益相关 $0.94/0.32/0.15$，
与 Rayleigh 功率增益理论 $J_0^2(2\pi f_D \Delta t)$ 吻合）。

> 若选 $900\ \mathrm{MHz}$，速度间相关性差异被压缩；若选 $28\ \mathrm{GHz}$，
> 即使低速时隙间也近乎独立，低速参照组失去意义。

**3. 与信道模型的标定频段一致。** TDL-C（3GPP TR 38.901）的时延扩展、角度谱参数在
$0.5\text{–}6\ \mathrm{GHz}$ Sub-6 场景下标定，$2.0\ \mathrm{GHz}$ 处于其保真区间。

**结论：** $f_c = 2\ \mathrm{GHz}$ 与 $\Delta t = 1\ \mathrm{ms}$ 是**配套设计**——
在此载波频率下，多普勒尺度使扫描的移动速度落在信道时变特性的敏感区间，
便于在差异化的时变衰落条件下评估功率控制策略。

In [2]:
C0 = 299_792_458.0  # 光速 [m/s]

@dataclass(frozen=True) #冻结这个类,不能修改里面的成员变量
class SceneConfig:
    # ---- 物理参数 ----
    f_c_hz: float = 2.0e9            # 载波频率 [Hz]

    # ---- 几何：直线道路，静止接收节点位于横向偏移 d_perp 处 ----
    d_perp_m: float = 20.0           # 道路到节点的垂直距离 [m]

    # ---- 时隙结构 ----
    dt_s: float = 1.0e-3             # 决策时隙 1 ms（保证低速时隙间信道强相关）
    T: int = 1000                    # episode 时隙数（共 1 s，最近点在 episode 中点）

    # ---- 速度场景 [km/h]：低 / 中 / 高 ----
    speeds_kmh: tuple = (30.0, 120.0, 350.0)

    # ---- 大尺度 ----
    shadow_sigma_db: float = 4.0     # 阴影衰落 std [dB]（每条轨迹内恒定）

    # ---- 功率 / 速率 ----
    # 选 noise=1e-8, avg=0.05, p_max=0.2：中位 SNR≈-2dB（信道受限区），
    # 注水相对均匀分配增益约 +20%，三组对比有明显区分度。
    bandwidth_hz: float = 10.0e6     # 带宽 [Hz]
    noise_power: float = 1.0e-8      # 等效接收噪声功率 [W]
    p_max: float = 0.2               # 单时隙最大发射功率 [W]
    avg_power: float = 0.05          # 平均功率预算 [W] → 总能量 E = avg_power * T * dt
    n_power_levels: int = 5          # 离散功率档位数（含 0 档），对应 5G TPC 式离散功率控制
    g_history_len: int = 8           # 观测中最近信道增益历史长度（frame stack，模拟 CSI 滤波/预测）


    # ---- 小尺度（Sionna TDL, TR 38.901）----
    tdl_profile: str = "C"           # A-E 可伸缩 profile；C = NLOS 富散射
    tdl_delay_spread_s: float = 300.0e-9   # TDL-C 名义 RMS 时延扩展 [s]
    num_sinusoids: int = 20          # TDL SoS 多普勒仿真正弦波数

    # ---- 数据规模 ----
    n_train: int = 1000              # 每速度训练轨迹数
    n_test: int = 100                # 每速度测试轨迹数（固定种子，三组方法共享）
    seed: int = 2026                 # 基础随机种子

    # ================= 派生量 =================
    def speed_to_mps(self, kmh: float) -> float:
        return kmh / 3.6

    def f_doppler(self, speed_kmh: float) -> float:
        """最大多普勒频移 [Hz]：f_D = v·f_c/c"""
        return self.speed_to_mps(speed_kmh) * self.f_c_hz / C0

    def episode_s(self) -> float:
        return self.T * self.dt_s

    def distance_m(self, speed_kmh: float) -> np.ndarray:
        """episode 内各时隙的车-节点距离 [T]（最近点位于 episode 中点）"""
        v = self.speed_to_mps(speed_kmh)
        t = np.arange(self.T) * self.dt_s
        x = v * (t - (self.T - 1) * self.dt_s / 2.0)
        return np.sqrt(self.d_perp_m ** 2 + x ** 2)

    def fspl_db(self, d_m) -> np.ndarray:
        """自由空间路径损耗 [dB]"""
        return 20.0 * np.log10(4.0 * np.pi * np.asarray(d_m) * self.f_c_hz / C0)


## 大尺度衰落：阴影衰落 $\sigma_{\mathrm{shadow}} = 4.0\ \mathrm{dB}$

### 1. 物理起源

无线信号在传播路径上遇到建筑物、地形、植被等**障碍物**时会产生随机遮挡衰减。
这种衰减的空间变化尺度为米~百米量级（远大于载波波长 $\lambda$），
因此称为**大尺度衰落**或**阴影衰落（shadow fading）**。

与小尺度衰落（多径干涉，厘米量级快变）不同，阴影衰落描述的是
**同一距离下接收功率的随机波动**——它是路径损耗 $L_{\mathrm{fs}}(d)$
无法解释的那部分大尺度随机成分。

### 2. 对数正态模型

阴影衰落的标准模型是**对数正态分布**（接收功率的 dB 值服从高斯分布）：

$$\xi \sim \mathcal{N}(0,\, \sigma_{\mathrm{shadow}}^2), \qquad
g_{\mathrm{shadow}} = 10^{\xi/10}$$

其中 $\xi$ 以 dB 为单位，$\sigma_{\mathrm{shadow}}$ 是其标准差（本实验取 $4.0$ dB）。
之所以用 dB 域的高斯分布，是因为大量独立障碍物的遮挡效应相乘后，
取对数由中心极限定理趋于正态（dB = $10\log_{10}$ 把乘法变加法）。

### 3. 在信道增益中的位置

最终线性信道增益 $g_t$ 由三部分相乘：

$$g_t = \underbrace{10^{-L_{\mathrm{fs}}(d(t))/10}}_{\text{路径损耗（确定性）}}
       \;\cdot\; \underbrace{10^{\xi/10}}_{\text{阴影衰落（每条轨迹恒定）}}
       \;\cdot\; \underbrace{g_{\mathrm{small}}(t)}_{\text{小尺度衰落（毫秒级快变）}}$$

| 组成 | 变化尺度 | 本实验中的处理 |
|---|---|---|
| 路径损耗 $L_{\mathrm{fs}}(d)$ | 随距离慢变 | $d(t)$ 的 U 形曲线决定 |
| 阴影衰落 $10^{\xi/10}$ | 极慢（每条轨迹内恒定） | $\xi \sim \mathcal{N}(0, 4^2)$ dB，轨迹间随机 |
| 小尺度 $g_{\mathrm{small}}$ | 毫秒级快变 | Sionna TDL-C SoS 模型 |

> **为什么阴影在一条轨迹内恒定？** 车辆在 1 秒内仅移动 8–97 m，
> 障碍物环境（大尺度地貌）基本不变，因此同一 pass-by 内阴影可视为常数；
> 不同轨迹（不同路段/随机种子）的阴影独立采样。
> 这种"块恒定"建模在蜂窝系统仿真中是标准做法（3GPP TR 38.901）。

### 4. 典型取值

| 场景 | $\sigma_{\mathrm{shadow}}$ [dB] |
|---|---|
| 农村 / 开阔地 | 2 – 4 |
| 郊区 | 3 – 6 |
| 城市城区 | 4 – 8 |
| 高铁覆盖走廊（规划覆盖） | 2 – 4 |

本实验取 $4.0$ dB，覆盖城区与郊区之间，对高铁专网偏保守但合理。

### 5. 对实验的影响

- **轨迹多样性**：阴影使每条轨迹的平均 SNR 整体偏移 $\pm$ 几 dB，
  训练/评估时策略面对不同链路预算，结果更鲁棒
- **注水增益**：阴影是乘性因子，对均匀分配与注水的速率影响相同
  （两者乘同一个 $g_{\mathrm{shadow}}$），因此**不改变注水相对均匀的增益比例**
  ——核心对比结论对 $\sigma_{\mathrm{shadow}}$ 不敏感
- **分布展宽**：评估速率的轨迹间方差主要来自阴影；增大 $\sigma$
  会让速率分布更宽，但均值关系不变


## 功率与速率参数：SNR 区间的协同标定

本组参数（$B$, $N_0$, $P_{\max}$, $\bar P$, 档位数, 历史长度）不是独立选取的，
而是一个**协同标定**：目标是将中位 SNR 锁定在**信道受限区**（约 $-5$ 至 $+5$ dB），
使注水相对均匀分配有 $15\text{–}25\%$ 的增益——区分度足够大，三组对比才有意义。

### 1. 带宽 $B = 10\ \mathrm{MHz}$

5G NR FR1 的标准带宽配置（TS 38.104 表 5.3.5-1），也是 LTE 单载波典型值。
决定香农速率的尺度因子：

$$R \approx B\log_2(1 + \mathrm{SNR})$$

$10\ \mathrm{MHz}$ 下每 dB SNR 约 $33\ \mathrm{Mbps}$ 理论上限，单链路速率落在 $10\text{–}30\ \mathrm{Mbps}$ 量级，
与真实 PDSCH 单流吞吐量对齐。

### 2. 噪声功率 $N_0 = 10^{-8}\ \mathrm{W} = -50\ \mathrm{dBm}$

由热噪声公式 $N = kTBF$ 出发：
$$kTB = 1.38\times 10^{-23} \times 290 \times 10^7 \approx 4\times 10^{-14}\ \mathrm{W} = -104\ \mathrm{dBm}$$
叠加 $10\ \mathrm{dB}$ 接收机噪声系数后约 $-94\ \mathrm{dBm}$。

本实验取 $-50\ \mathrm{dBm}$（等效抬高），目的是配合 $\bar P$ 将中位 SNR 压到目标区间。
物理上对应"低质量接收机或带外干扰较强"的场景。

### 3. 峰值功率 $P_{\max} = 0.2\ \mathrm{W} = 23\ \mathrm{dBm}$

5G UE Power Class 3 的典型最大发射功率（TS 38.101）。
与 $\bar P$ 的比值定义了**机会主义倍率**：

$$\eta = \frac{P_{\max}}{\bar P} = \frac{0.2}{0.05} = 4\quad(\text{好时隙最多发 4 倍平均功率})$$

这个倍率决定了策略的"动态范围"——$\eta$ 太小则注水和均匀无差别，
太大则策略容易在前期烧光预算。

### 4. 平均功率 $\bar P = 0.05\ \mathrm{W} = 17\ \mathrm{dBm}$ —— 标定核心

总能量预算：
$$E = \bar P \cdot T \cdot \Delta t = 0.05 \times 1000 \times 10^{-3} = 0.05\ \mathrm{J}$$

中位 SNR（决定实验是否落在"信道受限区"）：
$$\mathrm{SNR}_{\mathrm{med}} \approx 10\log_{10}\!\left(\frac{\bar P \cdot g_{\mathrm{med}}}{N_0}\right) \approx -2\ \mathrm{dB}$$

| SNR 区间 | 注水 vs 均匀增益 | 实验价值 |
|---|---|---|
| 高 SNR ($>10$ dB) | $<5\%$ | 注水无意义 |
| **中 SNR ($-5$ 至 $+5$ dB)** | **$15\text{–}25\%$** | **✅ 区分度最大** |
| 低 SNR ($<-10$ dB) | 理论大但绝对速率低 | 数值不直观 |

$17\ \mathrm{dBm}$ 对应真实 UE 的中等发射档位（手机日常通信典型功率），
使实验既落在注水增益最大的 SNR 区间，又对标真实 UE 功率规范。

### 5. 离散功率档位数 $N_{\mathrm{levels}} = 5$

5G PUCCH/PUSCH 功率控制用离散 TPC 命令（TS 38.213 表 7.1.1-1），
步长 $2\text{–}3$ dB。本实验 5 档为

$$\mathcal{P} = \{0,\ \bar P,\ 2\bar P,\ 3\bar P,\ 4\bar P\}
= \{0,\ 0.05,\ 0.1,\ 0.15,\ 0.2\}\ \mathrm{W}$$

步长 $0.05\ \mathrm{W} \approx 3\ \mathrm{dB}$，与 5G TPC 量级一致。

量化损失已实测：5 档相对连续注水的速率损失仅 $0.6\text{–}0.8\%$，
几乎不牺牲最优性，但大幅缩小动作空间（$|\mathcal{A}|=5$ vs 连续），
对策略学习友好。

### 6. 信道历史长度 $L_{\mathrm{hist}} = 8$（frame stack）

模拟 5G CSI 上报—生效延迟下的**滤波窗口**：
基站拿到的是过去若干时隙的 CSI 测量值，策略基于历史 $g$ 估计当前信道/趋势。

$8 \times \Delta t = 8\ \mathrm{ms}$ 的窗口在不同速度下覆盖不同的相关结构：

| 速度 | $T_c \approx 1/(2f_D)$ | 历史窗口/相干时间 |
|---|---|---|
| $30\ \mathrm{km/h}$ | $9\ \mathrm{ms}$ | $8/9 \approx 0.9$（近一个相干时间）|
| $350\ \mathrm{km/h}$ | $0.77\ \mathrm{ms}$ | $8/0.77 \approx 10$（约 10 个相干时间）|

> 高速下历史窗口远超相干时间，说明"用过去 g 预测当前 g"在高速时失效
> ——这正是因果策略在高速度下增益下降的物理解释（旧 CSI 已失相关）。

### 7. 四参数的协同关系

$$\frac{\bar P}{N_0} = \frac{0.05}{10^{-8}} = 5\times 10^6
\;\Rightarrow\; \mathrm{SNR}_{\mathrm{med}} \approx -2\ \mathrm{dB}
\;\Rightarrow\; \text{注水增益} \approx 15\text{–}20\%$$

$$\frac{P_{\max}}{\bar P} = 4 \;\Rightarrow\; \text{机会主义倍率} = 4$$

$$N_{\mathrm{levels}} = 5 \;\Rightarrow\; \text{量化损失} < 1\%$$

**一句话**：这组参数把实验锁定在
"信道受限区 + 4 倍动态范围 + 5G 离散功率档位"
的真实工程设定下，既保证注水有明显增益（让对比有意义），
又对标真实 UE 功率规范（让结论可迁移）。

## 小尺度信道模型：Sionna TDL-C 与仿真参数

### 1. TDL Profile 的选择（3GPP TR 38.901）

3GPP 定义了五种 TDL（Tapped Delay Line）轮廓 A--E，对应不同场景：

| Profile | 场景 | LoS/NLoS | 时延扩展 | 本实验? |
|---|---|---|---|---|
| A | 室内/微蜂窝 | LoS 强 | $<50$ ns | |
| B | 室内/微蜂窝 | LoS | $20$--$100$ ns | |
| **C** | **城市/郊区宏蜂窝** | **NLoS 富散射** | **$300$ ns** | **✅** |
| D | 城市 LoS | LoS | $300$ ns | |
| E | 城市高楼 | NLoS，大时延 | 大型 | |

选 **C** 的理由：
- 高铁场景典型 **NLoS**（车体/地形遮挡，富散射主导）
- 城市郊区宏蜂窝最贴近高铁沿线覆盖
- $300$ ns 在 $10$ MHz 带宽下满足**频率平坦衰落**条件（见下）

### 2. 时延扩展 $\tau_{\mathrm{rms}} = 300$ ns 与频率选择性

多径到达时间的统计宽度。$300$ ns 对应最大路径差约 $c\cdot\tau = 90$ m（城市建筑尺度反射体）。

相干带宽（信道近似不变的频率范围）：
$$B_c \approx \frac{1}{5\,\tau_{\mathrm{rms}}} = \frac{1}{5 \times 300\ \mathrm{ns}} \approx 667\ \mathrm{kHz} \ll B = 10\ \mathrm{MHz}$$

因 $B_c \ll B$ 时为频率选择性衰落，但本实验用**窄带单值模型**（5 条路径频率响应合成单值 $g_t$），
其物理依据是：在子载波级 $\Delta f = 15$ kHz $\ll B_c$ 内，信道频率响应近似恒定，
故可用一个 $g_t$ 代表整个子载波的增益。$300$ ns 是 TDL-C 的名义值（TR 38.901 表 7.5-6）。

### 3. SoS 正弦波数 $N_{\mathrm{sos}} = 20$

Sionna 用 **Sum-of-Sinusoids (SoS)** 生成瑞利衰落：
$$h(t) = \sum_{i=1}^{N} a_i\, e^{j(2\pi f_{D,i}\, t + \phi_i)}$$

- 理论上 $N\to\infty$ 才精确逼近 Jakes 多普勒谱
- 实践中 $N = 16$--$32$ 已足够；$20$ 是精度/速度的常用平衡点
- 仅影响轨迹生成的随机质量，不影响实验结论

## 数据规模与可复现性

| 参数 | 值 | 说明 |
|---|---|---|
| $N_{\mathrm{train}}$ | $1000$ | 每速度训练轨迹数 |
| $N_{\mathrm{test}}$ | $100$ | 每速度测试轨迹数（**固定种子，三组方法共享**）|
| seed | $2026$ | 基础随机种子，保证完全可复现 |

- 测试集固定种子 $\Rightarrow$ 均匀/注水/学习策略在**完全相同**的 $100$ 条轨迹上评估，对比公平
- $100$ 条测试轨迹的速率均值标准差约 $2$--$3\%$，足够分辨 $10\%+$ 的注水增益
- 总数据量：$3$ 速度 $\times 1100$ 轨迹 $\times 1000$ 时隙 $\approx 3.3$M 时隙

## 派生量与方法

### 多普勒频移

$$f_D = \frac{v\, f_c}{c}$$

| 速度 | $f_D$ | $f_D\,\Delta t$ | lag-1 相关（实测）|
|---|---|---|---|
| $30$ km/h | $55.6$ Hz | $0.056$ | $0.94$ |
| $120$ km/h | $222.2$ Hz | $0.222$ | $0.32$ |
| $350$ km/h | $648.1$ Hz | $0.648$ | $0.15$ |

### U 形距离曲线

$$d(t) = \sqrt{d_\perp^2 + x(t)^2}, \quad x(t) = v\!\left(t - \frac{T-1}{2}\Delta t\right)$$

- 以 $t = T/2$ 为对称中心（最近点 $d = d_\perp = 20$ m）
- 起止距离由速度决定：$30$ km/h 走 $8.3$ m（$d$: $20{\to}24{\to}20$），
  $350$ km/h 走 $97$ m（$d$: $52.5{\to}20{\to}52.5$）

### 自由空间路径损耗

$$L_{\mathrm{fs}}(d) = 20\log_{10}\!\left(\frac{4\pi d\, f_c}{c}\right)\ \mathrm{dB}$$

- $d = 20$ m 时约 $64$ dB，$d = 52.5$ m 时约 $73$ dB
- 动态范围约 $9$ dB，配合阴影 $\pm 4$ dB，总 SNR 波动约 $17$ dB

### 参数协同总结

| 协同关系 | 数值 |
|---|---|
| $\tau_{\mathrm{rms}} \ll 1/\Delta f$（窄带有效）| $300$ ns $\ll 67$ $\mu$s |
| $f_D\,\Delta t$ 跨相关区 | $0.056 / 0.222 / 0.648$ |
| $d_\perp$ 与速度范围配合 | $20$ m 偏移让 $350$ km/h 在 $1$s 内扫到 $52.5$ m |
| 测试集固定种子 | 三组方法公平对比 |